# Task 02 — Tabular features

**Wave 1** · target file: `backend/features.py` · prerequisites: none

**🎯 Goal:** `tabular_features(row)` — engineer the 16 baseline features (+ presence mask) from a raw catalog row.

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [1]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys
p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')
print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

repo root: /Users/appalonovaa/code/Macrocosm
models: ['fake_image_model.pkl']


## What to build
`tabular_features(row)` in `backend/features.py`. `row` is a dict of the 11 `RAW_TABULAR_FIELDS`.
Return **`(X, mask)`**, both length-16 float32 arrays in `TABULAR_FEATURES` order.

**Recipe** (this is exactly how the baseline was trained — KB *Preprocessing & features*):
1. **Clean SDSS sentinels**: any raw value `<= -100` (the -9999 'not measured' code) -> `NaN`.
2. **dered_u..z** — passthrough (5).
3. **colors** `g-r, u-g, r-i, i-z` = `dered_a - dered_b`, then **`clip(-1, 4)`** (4).
4. **log sizes** `log_<s>` = `log1p(max(<s>, 0))` for `expRad_r, deVRad_r, petroRad_r, petroR50_r, petroR90_r` (5).
5. **fracDeV_r** — passthrough (1).
6. **conc_r** = `petroR90_r / petroR50_r` (1).
7. **mask[i]** = 1.0 if feature i is finite else 0.0; then **fill non-finite values with 0.0** in `X`.

> The 16 features = `["dered_u","dered_g","dered_r","dered_i","dered_z","g-r","u-g","r-i","i-z","log_expRad_r","log_deVRad_r","log_petroRad_r","log_petroR50_r","log_petroR90_r","fracDeV_r","conc_r"]`.

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [ ]:
import numpy as np
SENTINEL = -100.0
TABULAR_FEATURES = ["dered_u","dered_g","dered_r","dered_i","dered_z","g-r","u-g","r-i","i-z","log_expRad_r","log_deVRad_r","log_petroRad_r","log_petroR50_r","log_petroR90_r","fracDeV_r","conc_r"]
RAW = ["dered_u","dered_g","dered_r","dered_i","dered_z","expRad_r","deVRad_r","petroRad_r","petroR50_r","petroR90_r","fracDeV_r"]
def tabular_features(row):
    clean = {}

    for name in RAW:
         value = float(row.get(name, np.nan))
         clean[name] = np.nan if value <= SENTINEL else value

    features = [
        clean['dered_u'],
        clean['dered_g'],
        clean['dered_r'],
        clean['dered_i'],
        clean['dered_z'],

        np.clip(clean['dered_g'] - clean['dered_r'], -1, 4),
        np.clip(clean['dered_u'] - clean['dered_g'], -1, 4),
        np.clip(clean['dered_r'] - clean['dered_i'], -1, 4),
        np.clip(clean['dered_i'] - clean['dered_z'], -1, 4),

        np.log1p(max(clean['expRad_r'], 0)),
        np.log1p(max(clean['deVRad_r'], 0)),
        np.log1p(max(clean['petroRad_r'], 0)),
        np.log1p(max(clean['petroR50_r'], 0)),
        np.log1p(max(clean['petroR90_r'], 0)),

        clean['fracDeV_r'],

        (
            clean['petroR90_r'] / clean['petroR50_r']
            if np.isfinite(clean['petroR90_r'])
            and np.isfinite(clean['petroR50_r'])
            and clean['petroR50_r'] != 0
            else np.nan
        )
    ]

    X = np.asarray(features, dtype=np.float32)
    mask = np.isfinite(X).astype(np.float32)

    X[~np.isfinite(X)] = 0.0

    return X, mask

    raise NotImplementedError

row = {"dered_u":19.,"dered_g":18.,"dered_r":17.5,"dered_i":17.2,"dered_z":17.,
       "expRad_r":1.2,"deVRad_r":1.1,"petroRad_r":2.0,"petroR50_r":1.5,"petroR90_r":3.5,"fracDeV_r":0.8}
X, mask = tabular_features(row)
assert X.shape == (16,) and mask.shape == (16,)
assert abs(X[0] - 19.) < 1e-5            # dered_u passthrough
assert abs(X[5] - 0.5) < 1e-5            # g-r = dered_g - dered_r = 18 - 17.5
assert abs(X[15] - 3.5/1.5) < 1e-5       # conc_r = petroR90_r / petroR50_r
assert mask.sum() == 16                  # everything present here
X2, m2 = tabular_features({**row, "petroR50_r": -9999})   # sentinel -> conc unusable
assert m2[15] == 0 and X2[15] == 0
print("OK", X.round(3))

OK [19.    18.    17.5   17.2   17.     0.5    1.     0.3    0.2    0.788
  0.742  1.099  0.916  1.504  0.8    2.333]


## ➡️ Move it into `backend/features.py`
Once the cell above passes, put your implementation into **`backend/features.py`** (replace the `# TODO`). Then run the **Check** cell — it imports from `backend/` and verifies the real module.

## ✅ Check (imports from `backend/`)

In [10]:
import importlib, numpy as np, backend.features as F; importlib.reload(F)
row = {"dered_u":19.,"dered_g":18.,"dered_r":17.5,"dered_i":17.2,"dered_z":17.,
       "expRad_r":1.2,"deVRad_r":1.1,"petroRad_r":2.0,"petroR50_r":1.5,"petroR90_r":3.5,"fracDeV_r":0.8}
X, mask = F.tabular_features(row)
assert X.shape == (16,) and abs(X[15] - 3.5/1.5) < 1e-5 and mask.sum() == 16
print("tabular_features OK")

AttributeError: 'Settings' object has no attribute 'RAW_TABULAR_FIELDS'

> ⚠️ **02 + 03 both edit `backend/features.py`.** Branch off a fresh `2026.6.17` and keep *both* functions when merging — see `MERGE.md`.

## 🚀 Ship it

In [ ]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/02-features-tabular
!git add backend/features.py notebooks/tasks-2026-6-17/02-features-tabular/task.ipynb
!git commit -m "02-features-tabular: Tabular features"
!git push -u origin task/02-features-tabular
# then merge back into 2026.6.17 -> see MERGE.md in this folder